# **Survival Analysis in Ovarian Cancer Patients - R** <br>
The study aims to identify risk factors associated with survival rate of mortality, disease progression, and reccurence among patients with ovarian cancer <br>

**Population** : Ovarian cancer patients <br> 
**Exposure** : Age, Stadium, ECOG classification, Types of Operation, Residue, Histotype, Degree of cancer <br> 
**Outcome** : Survival Outcomes (Overall survival (OS), Progression-Free Survival (PFS), Disease-Free Survival (DFS))

<u>**Main research questions answered in this analysis**</u> : 
- What are the survival outcomes of patient with ovarian cancer? --> Kaplan Meier analysis 
- Do survival outcomes differ between patients with different risk factors?? --> Kaplan Meier analysis and Log Rank Test or Cox Regression test
- Which risk factors are independently associated with survival outcomes?  --> Cox Proportional Hazard Regression 


<u>**Variable names used in this code**</u> <br>
***Dependent Variable*** : 
- Time_Death (numerical)        : Time (months)  from diagnosis to death of any cause 
- Time_Progression (numerical)  : Time (months) from diagnosis to disease progression or death 
- Time_DFS (numerical)          : Time (months) from diagnosis to disease-free state or death 
- Event_Death (nominal)         : alive/loss to follow up or death
- Event_Progression (nominal)   : alive/loss to follow up or progressed
- Event_DFS (nominal)           : alive/loss to follow up or residive 

***Independent Variables*** : 
- Age (nominal)         : <60, >=60
- Stadium (nominal)     : stadium of cancer 
- ECOG (nominal)        : classification of tumor based on ECOG 
- Operation (nominal)   : types of operation conducted to resect the tumor 
- Residue (nominal)     : borders of tumor tissue left after operation
- Histotype (nominal)   : classification of the tumor hystology 
- Degree (nominal)      : severity of cancer based on a consensus 
- LVSI (nominal)        : severity of the involvement of lympahtic system <br> 
*(this variable is later dropped due to significant missing data)* 

<u>**Statistical Analysis and Data Visualization used in this study**</u>:
1. Descriptive Analysis 
    - Frequency tables, Histogram, Boxplots of all variables 
    - Contigency tables and barplots of Event Death, Progression, DFS grouped by independent variables 
    - Summary table of Time Death, Progression, DFS grouped by independent variables
2. Missing Data Analysis
    - Frequency table, and heatmap of missing data
    - Little's test to exclude the hypothesis of Missing Completely at Random (MCAR) 
    = Logistic regression of missing data and other variables to look for patterns of Missing at Random (MAR) or MNAR (Missing Not at Random)
3. Missing data handling 
    - Multiple imputation (50 set imputation)
3. Kaplan Meier Analysis
    - Kaplan Meier analysis (overall) 
    - Kaplan Meier analysis grouped by independent variables
    - Log Rank test and Cox regression test
3. Mutlivariate Analysis 
    - Cox Proportional Hazard Regression <br>
<br>
- *Kaplan meier graph was not shown if analysis was conducted from the imputed dataset*
- *For log rank test, there is no reliable way to pool results from imputed data sets. In this analysis, average mean/median across imputed data was conducted.*
- *For Cox Regression, the function pool() allows a reliable way to combine estimates while accounting for variance.* 
- *Therefore, despite log rank test being the more common method to evaluate difference of survival among groups, Cox regression is more reliable when imputed dataset was used*  
- *adjustments have been made to protect confidentiality of the data*

**Data Cleaning**

In [ ]:
library(rio)
library(dplyr)
libary(tidyverse)
setwd("insert your path here")
raw <- import("./thefilename.csv")

library(tidyverse)

# 1. Cleaning data

# Convert 9999 to missing values
raw <- raw %>%
  mutate(across(everything(), ~ na_if(.x, 9999)))

# Convert variables to appropriate data types
raw <- raw %>%
  mutate(
    Usia = as.numeric(Usia),
    across(
      c(Paritas, STADIUM, ECOG, OPERASI, Residu, Histotipe,
        Derajat, LVSI, Sitologi, Kemoterapi, ResponsTerapi,
        DEATH, PROGRESIF, RESIDIF),
      as.factor
    )
  )


# 2. Converting dates

raw <- raw %>%
  mutate(
    DateofDiagnosis = as.Date(DateofDiagnosis, format = "%m/%d/%Y"),
    TGLRESPONS = as.Date(TGLRESPONS, format = "%m/%d/%Y"),
    DeathDate = as.Date(DeathDate, format = "%m/%d/%Y"),
    ProgressiveDate = as.Date(ProgressiveDate, format = "%m/%d/%Y"),
    ResidifDate = as.Date(ResidifDate, format = "%m/%d/%Y")
  )

# Checking how many dates failed to convert
raw %>%
  summarise(
    across(
      c(DateofDiagnosis, TGLRESPONS, DeathDate,
        ProgressiveDate, ResidifDate),
      ~ sum(is.na(.)),
      .names = "{.col}_missing"
    )
  )


# 3. Checking date inconsistencies

# Calculate preliminary time intervals in days
raw <- raw %>%
  mutate(
    TimeofDeath = as.numeric(DeathDate - DateofDiagnosis),
    TimeofProgression = as.numeric(ProgressiveDate - DateofDiagnosis),
    TimeofDFS = as.numeric(ResidifDate - TGLRESPONS)
  )

# Checking negative time intervals
raw %>%
  filter(TimeofDeath < 0) %>%
  select(DateofDiagnosis, DeathDate, TimeofDeath)

raw %>%
  filter(TimeofProgression < 0) %>%
  select(DateofDiagnosis, ProgressiveDate, TimeofProgression)

raw %>%
  filter(TimeofDFS < 0) %>%
  select(TGLRESPONS, ResidifDate, TimeofDFS)

# Checking progression occurring after death
raw %>%
  filter(TimeofProgression > TimeofDeath) %>%
  select(
    DateofDiagnosis,
    DeathDate,
    ProgressiveDate,
    TimeofDeath,
    TimeofProgression
  )


# 4. Correcting known data errors

# Sample 116: incorrect progression date
raw <- raw %>%
  mutate(
    ProgressiveDate = if_else(
      row_number() == 116,
      as.Date("2014-08-13"),
      ProgressiveDate
    )
  )

# Sample 60: incorrect death date
raw <- raw %>%
  mutate(
    DeathDate = if_else(
      row_number() == 60,
      as.Date("2020-11-04"),
      DeathDate
    )
  )


# 5. Recalculating survival times

raw <- raw %>%
  mutate(
    TimeofDeath = as.numeric(DeathDate - DateofDiagnosis),
    TimeofProgression = as.numeric(ProgressiveDate - DateofDiagnosis),
    TimeofDFS = as.numeric(ResidifDate - TGLRESPONS)
  )

# Final consistency checks
raw %>%
  summarise(
    Negative_Death = sum(TimeofDeath < 0, na.rm = TRUE),
    Negative_Progression = sum(TimeofProgression < 0, na.rm = TRUE),
    Negative_DFS = sum(TimeofDFS < 0, na.rm = TRUE),
    Progression_After_Death = sum(
      TimeofProgression > TimeofDeath,
      na.rm = TRUE
    )
  )


# 6. Creating independent variables

raw <- raw %>%
  mutate(
    
    # Age grouped into <60 and >=60
    Cat_Usia = factor(
      if_else(Usia >= 60, ">=60", "<60"),
      levels = c("<60", ">=60")
    ),
    
    # Stadium divided into I-II and III-IV
    Cat_Stadium = factor(
      if_else(
        as.numeric(as.character(STADIUM)) < 3,
        "I-II",
        "III-IV"
      ),
      levels = c("I-II", "III-IV")
    ),
    
    # ECOG grouped into 0 and >0
    Cat_ECOG = factor(
      if_else(
        as.numeric(as.character(ECOG)) == 0,
        "0",
        ">0"
      ),
      levels = c("0", ">0")
    ),
    
    # Types of operation
    Cat_Operasi = factor(
      OPERASI,
      levels = c(0, 1, 2),
      labels = c(
        "Surgical Staging",
        "Debulking",
        "Other"
      )
    ),
    
    # Residue grouped into <1 cm and >=1 cm
    Cat_Residu = factor(
      Residu,
      levels = c(0, 1),
      labels = c("<1 cm", ">=1 cm")
    ),
    
    # Histotype grouped into Serous and Non-Serous
    Cat_Histotipe = factor(
      if_else(
        Histotipe == 0,
        "Serous",
        "Non-Serous"
      ),
      levels = c("Serous", "Non-Serous")
    ),
    
    # Severity grouped into 1, 2, and 3
    Cat_Derajat = factor(
      Derajat,
      levels = c(0, 1, 2),
      labels = c("1", "2", "3")
    ),
    
    # LVSI grouped into negative and positive
    Cat_LVSI = factor(
      if_else(
        LVSI == 0,
        "Negatif",
        "Positif"
      ),
      levels = c("Negatif", "Positif")
    ),
    
    # Death status
    Event_Death = factor(
      if_else(
        DEATH == 0,
        "Hidup/LTF",
        "Mati"
      ),
      levels = c("Hidup/LTF", "Mati")
    ),
    
    # Progression status
    Event_Progression = factor(
      if_else(
        PROGRESIF == 0,
        "NED/LTF",
        "Progresif"
      ),
      levels = c("NED/LTF", "Progresif")
    ),
    
    # Recurrence status
    Event_DFS = factor(
      if_else(
        RESIDIF == 0,
        "NED/LTF",
        "Residif"
      ),
      levels = c("NED/LTF", "Residif")
    )
  )


# 7. Creating final analysis dataset

dat <- raw %>%
  transmute(
    Time_Death = TimeofDeath / 30,
    Event_Death,
    
    Time_Progression = TimeofProgression / 30,
    Event_Progression,
    
    Time_DFS = TimeofDFS / 30,
    Event_DFS,
    
    Age = Cat_Usia,
    Stadium = Cat_Stadium,
    ECOG = Cat_ECOG,
    Operation = Cat_Operasi,
    Residue = Cat_Residu,
    Histotype = Cat_Histotipe,
    Degree = Cat_Derajat,
    LVSI = Cat_LVSI
  )

**Descriptive Analysis**

*Frequency Tables*

In [ ]:
#Creating table for each variables
cols <- names(dat)[1:14]

freq_tables <- lapply(cols, function(col) {
  x <- dat[[col]]
  
  if (is.factor(x)) {
    table(x, useNA = "ifany")
  } else {
    c(summary(x), NA_Count = sum(is.na(x)))
  }
})

names(freq_tables) <- cols
freq_tables

*Histogram*

In [ ]:
#Plotting numerical variable (in months)
hist(dat$Time_Death, xlab ="Time - Death (bulan)", ylab ="Frekuensi", ylim = c(0,80), main = "Time of Death")
hist(dat$`Time_Progression`, xlab ="Time - Progression (bulan)", ylab ="Frekuensi", ylim = c(0,80), main = "Time of Progression")
hist(dat$`Time_DFS`, xlab ="Time - DFS (bulan)", ylab ="Frekuensi", ylim = c(0,80), main = "Time of DFS")

#Overlayed plots
hist(dat$Time_Death,
     col = rgb(1, 0, 0, 0.4),    #red
     ylim = c(0, 80),
     xlab = "Time (bulan)",
     ylab = "Frekuensi",
     main = "Overlayed Version")

hist(dat$Time_Progression,
     col = rgb(0, 1, 0, 0.4),   # green
     add = TRUE)

hist(dat$Time_DFS,
     col = rgb(0, 0, 1, 0.4),   # blue
     add = TRUE)

legend("topright",
       legend = c("Death", "Progression", "DFS"),
       fill = c(rgb(1,0,0,0.4), rgb(0,1,0,0.4), rgb(0,0,1,0.4)))

#Plotting categorical variables 
plot(dat$Event_Death, ylab = "Frekuensi", main= "Event-Death")
plot(dat$Event_Progression, ylab = "Frekuensi", main= "Event-Progression")
plot(dat$Event_DFS, ylab = "Frekuensi", main= "Event-DFS")


barplot(table(dat$Age, useNA = "ifany"), ylab = "Frekuensi", main= "Usia", ylim = c(0, max(table(dat$Age, useNA = "ifany")) + 20))
barplot(table(dat$Stadium, useNA = "ifany"), ylab = "Frekuensi", main= "Stadium", ylim = c(0, max(table(dat$Stadium, useNA = "ifany")) + 20))
barplot(table(dat$ECOG, useNA = "ifany"), ylab = "Frekuensi", main= "ECOG", ylim = c(0, max(table(dat$ECOG, useNA = "ifany")) + 20), names.arg = c("0", ">0", "NA"))
barplot(table(dat$Operation, useNA = "ifany"), ylab = "Frekuensi", main= "Jenis Operasi", ylim = c(0, max(table(dat$Operation, useNA = "ifany")) + 20), names.arg = c("Surg Stag", "Debulk", "Other"))
barplot(table(dat$Residue, useNA = "ifany"), ylab = "Frekuensi", main= "Residu", ylim = c(0, max(table(dat$Residue, useNA = "ifany")) + 20), names.arg = c("<1 cm", ">=1cm", "NA"))
barplot(table(dat$Histotype, useNA = "ifany"), ylab = "Frekuensi", main= "Histotipe", ylim = c(0, max(table(dat$Histotype, useNA = "ifany")) + 20))
barplot(table(dat$Degree, useNA = "ifany"), ylab = "Frekuensi", main= "Degree", ylim = c(0, max(table(dat$Degree, useNA = "ifany")) + 20), names.arg = c("1", "2", "3", "NA"))
barplot(table(dat$LVSI, useNA = "ifany"), ylab = "Frekuensi", main= "LVSI", ylim = c(0, max(table(dat$LVSI, useNA = "ifany")) + 20), names.arg = c("negatif", "positif", "NA"))


*Boxplots*

In [ ]:
#Creating Boxplots
boxplot(dat$Time_Death,
        dat$Time_Progression,
        dat$Time_DFS,
        names = c("Death", "Progression", "DFS"),
        main = "Comparison of Time Variables",
        ylab = "Time (bulan)",
        col = c("red", "green", "blue"))


*Frequency Tables of Event Death, Progression, and Remission grouped by Independent Variables*

In [ ]:
vars <- c("Age","Stadium","ECOG","Operation","Residue",
          "Histotype","Degree","LVSI")


# Contingency tables for Event_Death
cont_tablesDeath <- lapply(vars, function(v) {
  table(dat[[v]], dat$Event_Death, useNA = "ifany")
})

names(cont_tablesDeath) <- vars
cont_tablesDeath


# Contingency tables for Event_Progression
cont_tablesProg <- lapply(vars, function(v) {
  table(dat[[v]], dat$Event_Progression, useNA = "ifany")
})

names(cont_tablesProg) <- vars
cont_tablesProg


# Contingency tables for Event_DFS
cont_tablesDFS <- lapply(vars, function(v) {
  table(dat[[v]], dat$Event_DFS, useNA = "ifany")
})

names(cont_tablesDFS) <- vars
cont_tablesDFS

*Barplot of Event Death, Progression, Remission grouped by Independent Variables*

In [ ]:
#Barplot for Event_Death
for (v in vars) {
  barplot(
    table(dat$Event_Death, dat[[v]], useNA = "ifany"),
    beside = TRUE,
    col = c("lightblue", "pink"),
    legend = TRUE,
    xlab = v,
    ylab = "Count",
    main = paste("Event_Death by", v)
  )
}

#Barplot fot Event_Progression
for (v in vars) {
  barplot(
    table(dat$Event_Progression, dat[[v]], useNA = "ifany"),
    beside = TRUE,
    col = c("lightblue", "pink"),
    legend = TRUE,
    xlab = v,
    ylab = "Count",
    main = paste("Event_Progression by", v)
  )
}

#Barplot for Event_DFS
for (v in vars) {
  barplot(
    table(dat$Event_DFS, dat[[v]], useNA = "ifany"),
    beside = TRUE,
    col = c("lightblue", "pink"),
    legend = TRUE,
    xlab = v,
    ylab = "Count",
    main = paste("Event_DFS by", v)
  )
}

*Summary Table of OS, PFS, and DFS grouped by Independent Variables*  


In [ ]:
# Summary Table for Time_Death (no censoring included)
for (v in vars) {
  cat("\n")
  cat("Time_Death by:", v, "\n")
  
  result <- dat %>%
    group_by(.data[[v]]) %>%
    summarise(
      n = sum(!is.na(Time_Death)),
      mean = mean(Time_Death, na.rm = TRUE),
      sd = sd(Time_Death, na.rm = TRUE),
      median = median(Time_Death, na.rm = TRUE),
      min = min(Time_Death, na.rm = TRUE),
      max = max(Time_Death, na.rm = TRUE)
    )
  
  print(result)
}

# Summary Table for Time_Progression (no censoring included)
for (v in vars) {
  cat("\n")
  cat("Time_Progression by:", v, "\n")
  
  result <- dat %>%
    group_by(.data[[v]]) %>%
    summarise(
      n = sum(!is.na(Time_Progression)),
      mean = mean(Time_Progression, na.rm = TRUE),
      sd = sd(Time_Progression, na.rm = TRUE),
      median = median(Time_Progression, na.rm = TRUE),
      min = min(Time_Progression, na.rm = TRUE),
      max = max(Time_Progression, na.rm = TRUE)
    )
  
  print(result)
}

for (v in vars) {
  cat("\n")
  cat("Time_DFS by:", v, "\n")
  
  result <- dat %>%
    group_by(.data[[v]]) %>%
    summarise(
      n = sum(!is.na(Time_DFS)),
      mean = mean(Time_DFS, na.rm = TRUE),
      sd = sd(Time_DFS, na.rm = TRUE),
      median = median(Time_DFS, na.rm = TRUE),
      min = min(Time_DFS, na.rm = TRUE),
      max = max(Time_DFS, na.rm = TRUE)
    )
  
  print(result)
}

**Missing Data Analysis and Handling**

*Missing Data Analysis*

In [ ]:
install.packages("naniar")      
library(naniar)
miss_var_summary(dat)
vis_miss(dat)

#Little test for MCAR 
mcar_test(dat)

##Logistic Regression to see pattern of missingness
#Creating dummy variables for missingness indicator 
dat$ECOG_missing <- ifelse(is.na(dat$ECOG), 1, 0)
dat$Residue_missing <- ifelse(is.na(dat$Residue), 1, 0)
dat$Degree_missing <- ifelse(is.na(dat$Degree), 1, 0)
dat$LVSI_missing <- ifelse(is.na(dat$LVSI), 1, 0)

#ECOG
model <- glm(ECOG_missing ~ Time_Death + Event_Death + Time_Progression + Event_Progression + Time_DFS + Event_DFS + Age + Stadium + Operation + Histotype, 
             data = dat)
summary(model)

#Residue
model <- glm(Residue_missing ~ Time_Death + Event_Death + Time_Progression + Event_Progression + Time_DFS + Event_DFS + Age + Stadium + Operation + Histotype, 
             data = dat)
summary(model)

#Degree
model <- glm(Degree_missing ~ Time_Death + Event_Death + Time_Progression + Event_Progression + Time_DFS + Event_DFS + Age + Stadium + Operation + Histotype, 
             data = dat)
summary(model)

#LVSI 
model <- glm(LVSI_missing ~ Time_Death + Event_Death + Time_Progression + Event_Progression + Time_DFS + Event_DFS + Age + Stadium + Operation + Histotype, 
             data = dat)
summary(model)

#Deleting all dummy variable for missing data
dat <- dat %>% select(-any_of(c("ECOG_missing", "Residue_missing", "Degree_missing", "LVSI_missing")))


*Handling Missing Data by Multiple Imputation*

In [ ]:
##Multiple imputation
#We choose to exclude LVSI since >85% missing data
dat <- dat %>% select(-LVSI)

#Converting all Event factors from 1 2 to 0 1 for survival analysis purposes
dat$Event_Death <- ifelse(dat$Event_Death == "Mati", 1, 0)
dat$Event_Progression <- ifelse(dat$Event_Progression == "Progresif", 1, 0)
dat$Event_DFS <- ifelse(dat$Event_DFS == "Residif", 1, 0)

install.packages('mice')
library(mice)
md.pattern(dat, rotate.names = TRUE)

imp_data <- mice(data = dat, m = 50, maxit = 10, seed = 12345, print = FALSE)
imp_data
str(imp_data[1])
imp.datasets <-  complete(imp_data, "long")
imp.datasets

#Saving the imputed data into a file
write.csv(imp.datasets, file = "Completed Imputed Data.csv", row.names = FALSE)


**Kaplan Meier Analysis**

*Kaplan Meier Graph for Overall Survival (Death), Progression Free Survival, and Disease Free Survival (Remission)* 

In [ ]:
library(survival)

#Overall Survival
km.model <- survfit(Surv(dat$Time_Death, dat$Event_Death) ~ 1)
km.model
summary(km.model)

plot(km.model, 
     xlab = "Time (months)", 
     ylab ="%Alive - S(t)", 
     main = "KM Death", 
     las = 1, 
     mark.time = TRUE)
abline(h = 0.5, col = "red")

#Progression Free Survival
km_overall <- survfit(Surv(Time_Progression, Event_Progression) ~ 1, data = dat)
km_overall
summary(km_overall)

plot(km_overall,
     xlab = "Time (months)",
     ylab = "% Progression-Free - S(t)",
     main = "KM Progression-Free Survival",
     las = 1,
     mark.time = TRUE)
abline(h = 0.5, col = "red")

#Disease Free Survival
km_overall <- survfit(Surv(Time_DFS, Event_DFS) ~ 1, data = dat)
print(km_overall)
summary(km_overall)

plot(km_overall,
     xlab = "Time (months)",
     ylab = "% Disease-Free - S(t)",
     main = "KM Disease-Free Survival",
     las = 1,
     mark.time = TRUE)
abline(h = 0.5, col = "red")

*Kaplan Meier, Log Rank Test, and Cox Regression for Overall Survival grouped by Variables with no missing data*

- *Overall Survival*

In [ ]:
##Kaplan Meier, Log rank test, and Cox modelling for non missing variables (no need to use imputed data)
variables <- c("Age", "Stadium", "Operation", "Histotype")

for (var in variables) {
  
  cat("\n")
  cat("Variable:", var, "\n")
  
  formula <- as.formula(paste("Surv(Time_Death, Event_Death) ~", var))
  
  km.model <- survfit(formula, data = dat)
  
  print(km.model)
  print(summary(km.model))
  
  # Plot
  plot(km.model,
       xlab = "Time (months)",
       ylab = "%Alive - S(t)",
       main = paste("KM Death by", var),
       las = 1,
       col = 1:5,            
       mark.time = TRUE)
  
  abline(h = 0.5, col = "purple")
  
  legend("bottomright",
         legend = levels(as.factor(dat[[var]])),
         lty = 1,
         lwd = 2,
         col = 1:5,
         bty = "n",
         cex = 0.7)
  
  # Log-rank test
  print(survdiff(formula, data = dat))
  
  # Cox regression
  cox.model <- coxph(formula, data = dat)
  cox.summary <- summary(cox.model)
  
  HR <- round(exp(cox.summary$coef[, "coef"]), 3)
  Lower95 <- round(exp(cox.summary$coef[, "coef"] - 1.96*cox.summary$coef[, "se(coef)"]), 3)
  Upper95 <- round(exp(cox.summary$coef[, "coef"] + 1.96*cox.summary$coef[, "se(coef)"]), 3)
  p_value <- round(cox.summary$coef[, "Pr(>|z|)"], 4)
  
  result <- data.frame(HR = HR, Lower95 = Lower95, Upper95 = Upper95, p_value = p_value)
  print(result)
}

- *Progression Free Survival*

In [ ]:
##Kaplan Meier, Log rank test, and Cox modelling for non missing variables (no need to use imputed data)
variables <- c("Age", "Stadium", "Operation", "Histotype")

for (var in variables) {
  
  cat("\n")
  cat("Variable:", var, "\n")
  
  formula <- as.formula(paste("Surv(Time_Progression, Event_Progression) ~", var))
  
  km.model <- survfit(formula, data = dat)
  
  print(km.model)
  print(summary(km.model))
  
  # Plot
  plot(km.model,
       xlab = "Time (months)",
       ylab = "% Progression-Free - S(t)",
       main = paste("KM Progression by", var),
       las = 1,
       col = 1:5,            
       mark.time = TRUE)
  
  abline(h = 0.5, col = "purple")
  
  legend("bottomright",
         legend = levels(as.factor(dat[[var]])),
         lty = 1,
         lwd = 2,
         col = 1:5,
         bty = "n",
         cex = 0.7)
  
  # Log-rank test
  print(survdiff(formula, data = dat))
  
  # Cox regression
  cox.model <- coxph(formula, data = dat)
  cox.summary <- summary(cox.model)
  
  HR <- round(exp(cox.summary$coef[, "coef"]), 3)
  Lower95 <- round(exp(cox.summary$coef[, "coef"] - 1.96*cox.summary$coef[, "se(coef)"]), 3)
  Upper95 <- round(exp(cox.summary$coef[, "coef"] + 1.96*cox.summary$coef[, "se(coef)"]), 3)
  p_value <- round(cox.summary$coef[, "Pr(>|z|)"], 4)
  
  result <- data.frame(HR = HR, Lower95 = Lower95, Upper95 = Upper95, p_value = p_value)
  print(result)
}

- *Disease Free Survival*

In [ ]:
##Kaplan Meier, Log rank test, and Cox modelling for non missing variables (no need to use imputed data)
variables <- c("Age", "Stadium", "Operation", "Histotype")

for (var in variables) {
  
  cat("\n")
  cat("Variable:", var, "\n")
  
  formula <- as.formula(paste("Surv(Time_DFS, Event_DFS) ~", var))
  
  km.model <- survfit(formula, data = dat)
  
  print(km.model)
  print(summary(km.model))
  
  # Plot
  plot(km.model,
       xlab = "Time (months)",
       ylab = "% Disease-Free - S(t)",
       main = paste("KM DFS by", var),
       las = 1,
       col = 1:5,            
       mark.time = TRUE)
  
  abline(h = 0.5, col = "purple")
  
  legend("bottomright",
         legend = levels(as.factor(dat[[var]])),
         lty = 1,
         lwd = 2,
         col = 1:5,
         bty = "n",
         cex = 0.7)
  
  # Log-rank test
  print(survdiff(formula, data = dat))
  
  # Cox regression
  cox.model <- coxph(formula, data = dat)
  cox.summary <- summary(cox.model)
  
  HR <- round(exp(cox.summary$coef[, "coef"]), 3)
  Lower95 <- round(exp(cox.summary$coef[, "coef"] - 1.96*cox.summary$coef[, "se(coef)"]), 3)
  Upper95 <- round(exp(cox.summary$coef[, "coef"] + 1.96*cox.summary$coef[, "se(coef)"]), 3)
  p_value <- round(cox.summary$coef[, "Pr(>|z|)"], 4)
  
  result <- data.frame(HR = HR, Lower95 = Lower95, Upper95 = Upper95, p_value = p_value)
  print(result)
}

*Log Rank Test, and Cox Regression grouped by Variables with missing data (ECOG, Residue, and Degree*<br>
*Kaplan Meier graph is avoided in imputed variables to avoid misrepresentation*

- *Kaplan meier graph was not shown if analysis was conducted from imputed dataset*
- *For log rank test, there is no reliable way to pool results from imputed data sets. In this analysis, average mean/median across imputed data was conducted.*
- *For Cox Regression, the function pool() allows a reliable way to combine estimates while accounting for variance.* 
- *Therefore, despite log rank test being the more common method to evaluate difference of survival among groups, Cox regression is more reliable when imputed dataset was used*

- *Overall Survival (Death)*

In [ ]:
##########################
##ECOG 
fit_ECOG <- with(imp_data, survdiff(Surv(Time_Death, Event_Death) ~ ECOG))
chisq_ECOG <- sapply(fit_ECOG$analyses, function(x) x$chisq)
df_ECOG <- sapply(fit_ECOG$analyses, function(x) length(x$n) - 1)
p_ECOG <- pchisq(chisq_ECOG, df = df_ECOG, lower.tail = FALSE)

#Pooled Log rank test 
summary_ECOG <- data.frame(
  Variable   = "ECOG",
  Mean_ChiSq = round(mean(chisq_ECOG), 3),
  Median_p   = round(median(p_ECOG), 4),
  Mean_p     = round(mean(p_ECOG), 4),
  Min_p      = round(min(p_ECOG), 4),
  Max_p      = round(max(p_ECOG), 4)
)

summary_ECOG


#Pooled Cox regression test
fit_ECOG_cox <- with(imp_data, coxph(Surv(Time_Death, Event_Death) ~ ECOG))

pooled_ECOG <- pool(fit_ECOG_cox)

summary_ECOG_cox <- summary(pooled_ECOG)

summary_ECOG_cox <- data.frame(
  Variable = rownames(summary_ECOG_cox),
  Estimate = round(summary_ECOG_cox$estimate, 3),
  SE       = round(summary_ECOG_cox$std.error, 3),
  HR       = round(exp(summary_ECOG_cox$estimate), 3),
  Lower95  = round(exp(summary_ECOG_cox$estimate - 1.96*summary_ECOG_cox$std.error), 3),
  Upper95  = round(exp(summary_ECOG_cox$estimate + 1.96*summary_ECOG_cox$std.error), 3),
  p_value  = round(summary_ECOG_cox$p.value, 4)
)

summary_ECOG_cox


###############################
#Residue
fit_Residue <- with(imp_data, survdiff(Surv(Time_Death, Event_Death) ~ Residue))
chisq_Residue <- sapply(fit_Residue$analyses, function(x) x$chisq)
df_Residue <- sapply(fit_Residue$analyses, function(x) length(x$n) - 1)
p_Residue <- pchisq(chisq_Residue, df = df_Residue, lower.tail = FALSE)

#Pooled Log Rank test 
summary_Residue <- data.frame(
  Variable   = "Residue",
  Mean_ChiSq = round(mean(chisq_Residue), 3),
  Median_p   = round(median(p_Residue), 4),
  Mean_p     = round(mean(p_Residue), 4),
  Min_p      = round(min(p_Residue), 4),
  Max_p      = round(max(p_Residue), 4)
)

summary_Residue


#Pooled Cox regression test
fit_Residue_cox <- with(imp_data, coxph(Surv(Time_Death, Event_Death) ~ Residue))

pooled_Residue <- pool(fit_Residue_cox)
summary_Residue_cox <- summary(pooled_Residue)

summary_Residue_cox <- data.frame(
  Variable = rownames(summary_Residue_cox),
  Estimate = round(summary_Residue_cox$estimate, 3),
  SE       = round(summary_Residue_cox$std.error, 3),
  HR       = round(exp(summary_Residue_cox$estimate), 3),
  Lower95  = round(exp(summary_Residue_cox$estimate - 1.96*summary_Residue_cox$std.error), 3),
  Upper95  = round(exp(summary_Residue_cox$estimate + 1.96*summary_Residue_cox$std.error), 3),
  p_value  = round(summary_Residue_cox$p.value, 4)
)

summary_Residue_cox


###############################
#Degree
fit_Degree <- with(imp_data, survdiff(Surv(Time_Death, Event_Death) ~ Degree))
chisq_Degree <- sapply(fit_Degree$analyses, function(x) x$chisq)
df_Degree <- sapply(fit_Degree$analyses, function(x) length(x$n) - 1)
p_Degree <- pchisq(chisq_Degree, df = df_Degree, lower.tail = FALSE)

#Pooled Log Rank Test 
summary_Degree <- data.frame(
  Variable   = "Degree",
  Mean_ChiSq = round(mean(chisq_Degree), 3),
  Median_p   = round(median(p_Degree), 4),
  Mean_p     = round(mean(p_Degree), 4),
  Min_p      = round(min(p_Degree), 4),
  Max_p      = round(max(p_Degree), 4)
)

summary_Degree

#Pooled Cox regression test
fit_Degree_cox <- with(imp_data, coxph(Surv(Time_Death, Event_Death) ~ Degree))

pooled_Degree <- pool(fit_Degree_cox)

summary_Degree_cox <- summary(pooled_Degree)

summary_Degree_cox <- data.frame(
  Variable = rownames(summary_Degree_cox),
  Estimate = round(summary_Degree_cox$estimate, 3),
  SE       = round(summary_Degree_cox$std.error, 3),
  HR       = round(exp(summary_Degree_cox$estimate), 3),
  Lower95  = round(exp(summary_Degree_cox$estimate - 1.96*summary_Degree_cox$std.error), 3),
  Upper95  = round(exp(summary_Degree_cox$estimate + 1.96*summary_Degree_cox$std.error), 3),
  p_value  = round(summary_Degree_cox$p.value, 4)
)

summary_Degree_cox

- *Progression Free Survival*

In [ ]:
##########################
##ECOG 
fit_ECOG <- with(imp_data, survdiff(Surv(Time_Progression, Event_Progression) ~ ECOG))
chisq_ECOG <- sapply(fit_ECOG$analyses, function(x) x$chisq)
df_ECOG <- sapply(fit_ECOG$analyses, function(x) length(x$n) - 1)
p_ECOG <- pchisq(chisq_ECOG, df = df_ECOG, lower.tail = FALSE)

#Pooled Log rank test 
summary_ECOG <- data.frame(
  Variable   = "ECOG",
  Mean_ChiSq = round(mean(chisq_ECOG), 3),
  Median_p   = round(median(p_ECOG), 4),
  Mean_p     = round(mean(p_ECOG), 4),
  Min_p      = round(min(p_ECOG), 4),
  Max_p      = round(max(p_ECOG), 4)
)

summary_ECOG


#Pooled Cox regression test
fit_ECOG_cox <- with(imp_data, coxph(Surv(Time_Progression, Event_Progression) ~ ECOG))

pooled_ECOG <- pool(fit_ECOG_cox)

summary_ECOG_cox <- summary(pooled_ECOG)

summary_ECOG_cox <- data.frame(
  Variable = rownames(summary_ECOG_cox),
  Estimate = round(summary_ECOG_cox$estimate, 3),
  SE       = round(summary_ECOG_cox$std.error, 3),
  HR       = round(exp(summary_ECOG_cox$estimate), 3),
  Lower95  = round(exp(summary_ECOG_cox$estimate - 1.96*summary_ECOG_cox$std.error), 3),
  Upper95  = round(exp(summary_ECOG_cox$estimate + 1.96*summary_ECOG_cox$std.error), 3),
  p_value  = round(summary_ECOG_cox$p.value, 4)
)

summary_ECOG_cox


###############################
#Residue
fit_Residue <- with(imp_data, survdiff(Surv(Time_Progression, Event_Progression) ~ Residue))
chisq_Residue <- sapply(fit_Residue$analyses, function(x) x$chisq)
df_Residue <- sapply(fit_Residue$analyses, function(x) length(x$n) - 1)
p_Residue <- pchisq(chisq_Residue, df = df_Residue, lower.tail = FALSE)

#Pooled Log Rank test 
summary_Residue <- data.frame(
  Variable   = "Residue",
  Mean_ChiSq = round(mean(chisq_Residue), 3),
  Median_p   = round(median(p_Residue), 4),
  Mean_p     = round(mean(p_Residue), 4),
  Min_p      = round(min(p_Residue), 4),
  Max_p      = round(max(p_Residue), 4)
)

summary_Residue


#Pooled Cox regression test
fit_Residue_cox <- with(imp_data, coxph(Surv(Time_Progression, Event_Progression) ~ Residue))

pooled_Residue <- pool(fit_Residue_cox)
summary_Residue_cox <- summary(pooled_Residue)

summary_Residue_cox <- data.frame(
  Variable = rownames(summary_Residue_cox),
  Estimate = round(summary_Residue_cox$estimate, 3),
  SE       = round(summary_Residue_cox$std.error, 3),
  HR       = round(exp(summary_Residue_cox$estimate), 3),
  Lower95  = round(exp(summary_Residue_cox$estimate - 1.96*summary_Residue_cox$std.error), 3),
  Upper95  = round(exp(summary_Residue_cox$estimate + 1.96*summary_Residue_cox$std.error), 3),
  p_value  = round(summary_Residue_cox$p.value, 4)
)

summary_Residue_cox


###############################
#Degree
fit_Degree <- with(imp_data, survdiff(Surv(Time_Progression, Event_Progression) ~ Degree))
chisq_Degree <- sapply(fit_Degree$analyses, function(x) x$chisq)
df_Degree <- sapply(fit_Degree$analyses, function(x) length(x$n) - 1)
p_Degree <- pchisq(chisq_Degree, df = df_Degree, lower.tail = FALSE)

#Pooled Log Rank Test 
summary_Degree <- data.frame(
  Variable   = "Degree",
  Mean_ChiSq = round(mean(chisq_Degree), 3),
  Median_p   = round(median(p_Degree), 4),
  Mean_p     = round(mean(p_Degree), 4),
  Min_p      = round(min(p_Degree), 4),
  Max_p      = round(max(p_Degree), 4)
)

summary_Degree

#Pooled Cox regression test
fit_Degree_cox <- with(imp_data, coxph(Surv(Time_Progression, Event_Progression) ~ Degree))

pooled_Degree <- pool(fit_Degree_cox)

summary_Degree_cox <- summary(pooled_Degree)

summary_Degree_cox <- data.frame(
  Variable = rownames(summary_Degree_cox),
  Estimate = round(summary_Degree_cox$estimate, 3),
  SE       = round(summary_Degree_cox$std.error, 3),
  HR       = round(exp(summary_Degree_cox$estimate), 3),
  Lower95  = round(exp(summary_Degree_cox$estimate - 1.96*summary_Degree_cox$std.error), 3),
  Upper95  = round(exp(summary_Degree_cox$estimate + 1.96*summary_Degree_cox$std.error), 3),
  p_value  = round(summary_Degree_cox$p.value, 4)
)

summary_Degree_cox

- *Disease Free Survival*

In [ ]:
##########################
##ECOG 
fit_ECOG <- with(imp_data, survdiff(Surv(Time_DFS, Event_DFS) ~ ECOG))
chisq_ECOG <- sapply(fit_ECOG$analyses, function(x) x$chisq)
df_ECOG <- sapply(fit_ECOG$analyses, function(x) length(x$n) - 1)
p_ECOG <- pchisq(chisq_ECOG, df = df_ECOG, lower.tail = FALSE)

#Pooled Log rank test 
summary_ECOG <- data.frame(
  Variable   = "ECOG",
  Mean_ChiSq = round(mean(chisq_ECOG), 3),
  Median_p   = round(median(p_ECOG), 4),
  Mean_p     = round(mean(p_ECOG), 4),
  Min_p      = round(min(p_ECOG), 4),
  Max_p      = round(max(p_ECOG), 4)
)

summary_ECOG


#Pooled Cox regression test
fit_ECOG_cox <- with(imp_data, coxph(Surv(Time_DFS, Event_DFS) ~ ECOG))

pooled_ECOG <- pool(fit_ECOG_cox)

summary_ECOG_cox <- summary(pooled_ECOG)

summary_ECOG_cox <- data.frame(
  Variable = rownames(summary_ECOG_cox),
  Estimate = round(summary_ECOG_cox$estimate, 3),
  SE       = round(summary_ECOG_cox$std.error, 3),
  HR       = round(exp(summary_ECOG_cox$estimate), 3),
  Lower95  = round(exp(summary_ECOG_cox$estimate - 1.96*summary_ECOG_cox$std.error), 3),
  Upper95  = round(exp(summary_ECOG_cox$estimate + 1.96*summary_ECOG_cox$std.error), 3),
  p_value  = round(summary_ECOG_cox$p.value, 4)
)

summary_ECOG_cox


###############################
#Residue
fit_Residue <- with(imp_data, survdiff(Surv(Time_DFS, Event_DFS) ~ Residue))
chisq_Residue <- sapply(fit_Residue$analyses, function(x) x$chisq)
df_Residue <- sapply(fit_Residue$analyses, function(x) length(x$n) - 1)
p_Residue <- pchisq(chisq_Residue, df = df_Residue, lower.tail = FALSE)

#Pooled Log Rank test 
summary_Residue <- data.frame(
  Variable   = "Residue",
  Mean_ChiSq = round(mean(chisq_Residue), 3),
  Median_p   = round(median(p_Residue), 4),
  Mean_p     = round(mean(p_Residue), 4),
  Min_p      = round(min(p_Residue), 4),
  Max_p      = round(max(p_Residue), 4)
)

summary_Residue


#Pooled Cox regression test
fit_Residue_cox <- with(imp_data, coxph(Surv(Time_DFS, Event_DFS) ~ Residue))

pooled_Residue <- pool(fit_Residue_cox)
summary_Residue_cox <- summary(pooled_Residue)

summary_Residue_cox <- data.frame(
  Variable = rownames(summary_Residue_cox),
  Estimate = round(summary_Residue_cox$estimate, 3),
  SE       = round(summary_Residue_cox$std.error, 3),
  HR       = round(exp(summary_Residue_cox$estimate), 3),
  Lower95  = round(exp(summary_Residue_cox$estimate - 1.96*summary_Residue_cox$std.error), 3),
  Upper95  = round(exp(summary_Residue_cox$estimate + 1.96*summary_Residue_cox$std.error), 3),
  p_value  = round(summary_Residue_cox$p.value, 4)
)

summary_Residue_cox


###############################
#Degree
fit_Degree <- with(imp_data, survdiff(Surv(Time_DFS, Event_DFS) ~ Degree))
chisq_Degree <- sapply(fit_Degree$analyses, function(x) x$chisq)
df_Degree <- sapply(fit_Degree$analyses, function(x) length(x$n) - 1)
p_Degree <- pchisq(chisq_Degree, df = df_Degree, lower.tail = FALSE)

#Pooled Log Rank Test 
summary_Degree <- data.frame(
  Variable   = "Degree",
  Mean_ChiSq = round(mean(chisq_Degree), 3),
  Median_p   = round(median(p_Degree), 4),
  Mean_p     = round(mean(p_Degree), 4),
  Min_p      = round(min(p_Degree), 4),
  Max_p      = round(max(p_Degree), 4)
)

summary_Degree

#Pooled Cox regression test
fit_Degree_cox <- with(imp_data, coxph(Surv(Time_DFS, Event_DFS) ~ Degree))

pooled_Degree <- pool(fit_Degree_cox)

summary_Degree_cox <- summary(pooled_Degree)

summary_Degree_cox <- data.frame(
  Variable = rownames(summary_Degree_cox),
  Estimate = round(summary_Degree_cox$estimate, 3),
  SE       = round(summary_Degree_cox$std.error, 3),
  HR       = round(exp(summary_Degree_cox$estimate), 3),
  Lower95  = round(exp(summary_Degree_cox$estimate - 1.96*summary_Degree_cox$std.error), 3),
  Upper95  = round(exp(summary_Degree_cox$estimate + 1.96*summary_Degree_cox$std.error), 3),
  p_value  = round(summary_Degree_cox$p.value, 4)
)

summary_Degree_cox

**Multivariate Analysis**

 - *Cox Proportional Hazard Regression*<br>
 **only variables with p value < 0.25 from bivariate cox regression was included in the model* 

In [ ]:
selected_vars <- c("ECOG","Operation","Histotype")
fit_multi <- with(imp_data, coxph(as.formula(paste("Surv(Time_DFS, Event_DFS) ~", paste(selected_vars, collapse = "+")))))
pooled_multi <- pool(fit_multi)
s_multi <- summary(pooled_multi)

s_multi$HR <- exp(s_multi$estimate)
s_multi$Lower95 <- exp(s_multi$estimate - 1.96*s_multi$std.error)
s_multi$Upper95 <- exp(s_multi$estimate + 1.96*s_multi$std.error)

# Final multivariate table
multivar_table <- s_multi[, c("term","estimate","std.error","statistic","df","p.value","HR","Lower95","Upper95")]
print(multivar_table)

**saving all environments and values for reproducibility**

In [ ]:
save.image("Environment_survival.Rdata")